In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GraphSAGE
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import negative_sampling


ModuleNotFoundError: No module named 'torch_geometric'

In [ ]:


# ===== CSV: columns src,dst (любой int id) =====
df = pd.read_csv("edges.csv")[["src", "dst"]]

# remap id -> 0..N-1 (коротко)
nodes = pd.Index(df["src"]).append(pd.Index(df["dst"])).unique()
id2new = {k: i for i, k in enumerate(nodes)}
src = torch.tensor(df["src"].map(id2new).to_numpy(), dtype=torch.long)
dst = torch.tensor(df["dst"].map(id2new).to_numpy(), dtype=torch.long)
edge_index = torch.stack([src, dst], dim=0)
num_nodes = len(nodes)

data = Data(edge_index=edge_index, num_nodes=num_nodes)

# split edges
train_data, val_data, test_data = RandomLinkSplit(
    num_val=0.05, num_test=0.10,
    is_undirected=False,
    add_negative_train_samples=False,
)(data)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_data, val_data, test_data = train_data.to(device), val_data.to(device), test_data.to(device)

# GraphSAGE encoder (готовая реализация)
model = GraphSAGE(
    in_channels=64, hidden_channels=128, num_layers=2, out_channels=128
).to(device)

# если нет признаков узлов — используем обучаемые эмбеддинги как x
x = torch.nn.Embedding(num_nodes, 64).to(device)

opt = torch.optim.Adam(list(model.parameters()) + list(x.parameters()), lr=1e-3)

def decode_dot(z, edge_label_index):
    s, t = edge_label_index
    return (z[s] * z[t]).sum(dim=-1)

@torch.no_grad()
def auc(split):
    z = model(x.weight, train_data.edge_index)
    logits = decode_dot(z, split.edge_label_index)
    probs = torch.sigmoid(logits)
    return float(binary_auroc(probs, split.edge_label.int()))

def train():
    model.train(); x.train()
    opt.zero_grad()
    z = model(x.weight, train_data.edge_index)

    pos = train_data.edge_index
    neg = negative_sampling(train_data.edge_index, num_nodes=num_nodes, num_neg_samples=pos.size(1))

    logits = torch.cat([decode_dot(z, pos), decode_dot(z, neg)])
    labels = torch.cat([torch.ones(pos.size(1)), torch.zeros(neg.size(1))]).to(device)

    loss = F.binary_cross_entropy_with_logits(logits, labels)
    loss.backward(); opt.step()
    return float(loss)

for e in range(1, 21):
    loss = train()
    print(e, "loss", round(loss, 4), "val_auc", round(auc(val_data), 4))

print("test_auc", round(auc(test_data), 4))